In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# Sample dataset
data = [
    ("i am learning", "je suis en train d apprendre"),
    ("he is running", "il court"),
    ("she is happy", "elle est heureuse"),
    ("i am happy", "je suis heureux")
]

input_texts = [x[0] for x in data]

target_texts = ["<start> " + x[1] + " <end>" for x in data]

# ***Tokenization***

In [ ]:
#Encoder Tokenizer
input_tokenizer=Tokenizer()
input_tokenizer.fit_on_texts(input_texts)
input_sequences=input_tokenizer.texts_to_sequences(input_texts)

#Decoder Tokenizer
target_tokenizer=Tokenizer(filters='')
target_tokenizer. fit_on_texts(target_texts)
target_sequences=target_tokenizer. texts_to_sequences(target_texts)

## ***Padding***

In [ ]:
max_input_len=max([len(seq) for seq in input_sequences])
max_target_len=max([len(seq) for seq in target_sequences])

encoder_input_data=pad_sequences(input_sequences, maxlen=max_input_len, padding='post')
decoder_input_data=pad_sequences(target_sequences,maxlen=max_target_len, padding='post')

# Create Decoder Target (Shifted)

Important teaching point:

Decoder input = ...

Decoder output = ...

In [ ]:
decoder_target_data=np.zeros_like(decoder_input_data)

for i in range(len(data)):
  decoder_target_data[i, :- 1]=decoder_input_data[i,1:]

# ***Build Model***

In [ ]:
#Parameters
vocab_size_input=len(input_tokenizer.word_index)+1
vocab_size_target=len(target_tokenizer.word_index)+1
embedding_dim=256
latent_dim=256

# ***Encoder***

In [ ]:
encoder_inputs=Input(shape=(None,))
enc_emb=Embedding(vocab_size_input, embedding_dim) (encoder_inputs)

encoder_lstm=LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)
#Encoder understands meaning and stores it in state_h, start_c, ]These togeather from Context
encoder_states = [state_h, state_c]
print(encoder_states)

[<KerasTensor shape=(None, 256), dtype=float32, sparse=False, ragged=False, name=keras_tensor_50>, <KerasTensor shape=(None, 256), dtype=float32, sparse=False, ragged=False, name=keras_tensor_51>]


#***Decoder***

In [ ]:
decoder_inputs=Input(shape=(None,))
dec_emb_layer=Embedding(vocab_size_target, embedding_dim)
dec_emb=dec_emb_layer(decoder_inputs)

decoder_lstm=LSTM(latent_dim,return_sequences=True,return_state=True)
decoder_outputs,_,_= decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense=Dense(vocab_size_target,activation='softmax')
decoder_outputs=decoder_dense(decoder_outputs)
print(decoder_outputs)

<KerasTensor shape=(None, None, 15), dtype=float32, sparse=False, ragged=False, name=keras_tensor_57>


# ***Compile Model***

In [ ]:

model=Model([encoder_inputs, decoder_inputs ], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

# ***Train Model***

In [ ]:


model.fit(
[encoder_input_data, decoder_input_data],
decoder_target_data,
batch_size=1,
epochs=200,
validation_split=0.2

)

Epoch 1/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 274ms/step - loss: 2.6966 - val_loss: 2.6121
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 2.5559 - val_loss: 2.4840
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 2.3502 - val_loss: 2.2541
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 2.0419 - val_loss: 1.8008
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 1.6949 - val_loss: 1.4875
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - loss: 1.6507 - val_loss: 1.4745
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 1.6165 - val_loss: 1.5606
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 1.4798 - val_loss: 1.4809
Epoch 9/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - loss: 1.4089 - val_loss: 1.4465
Epoch 10/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 1.3911 - val_loss: 1.4536
Epoch 11/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - loss: 1.3598 - val_loss: 1.5054
Epoch 12/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 1.2730 - val_l

# ***Inference Model***

Training model != Prediction model

We build seperate inference models

# ***Encoder Interface***

In [ ]:
encoder_model=Model(encoder_inputs,encoder_states)

In [ ]:
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_emb2 = Embedding(vocab_size_target, embedding_dim)(decoder_inputs)

decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2, initial_state=decoder_states_inputs
)

decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2
)

# ***Prediction Function***

In [ ]:
reverse_target_index = {i: word for word, i in target_tokenizer.word_index.items()}

def decode_sequence(input_seq):

    states_value = encoder_model.predict(input_seq)

    start_token = target_tokenizer.word_index.get('<start>')

    if start_token is None:
        start_token = target_tokenizer.word_index.get('start')

    end_token = target_tokenizer.word_index.get('<end>')

    if end_token is None:
        end_token = target_tokenizer.word_index.get('end')

    target_seq = np.array([[start_token]])

    stop_condition = False
    decoded_sentence = ""

    while not stop_condition:

        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value
        )

        sampled_token_index = np.argmax(output_tokens[0, -1, :])

        sampled_word = reverse_target_index.get(
            sampled_token_index, ''
        )

        if (
            sampled_token_index == end_token or
            len(decoded_sentence.split()) > max_target_len
        ):
            stop_condition = True
        else:
            decoded_sentence += " " + sampled_word

        target_seq = np.array([[sampled_token_index]])

        states_value = [h, c]

    return decoded_sentence

# ***Test***

In [ ]:
test_input = "i will kill you"
seq = input_tokenizer.texts_to_sequences([test_input])
seq = pad_sequences(seq, maxlen=max_input_len, padding='post')

print("Input:", test_input)
print("Output:", decode_sequence(seq))

Input: i will kill you
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
Output:  je suis en train d apprendre
